# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadBilalFarooq/Assignment1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
import pandas as pd

url = "https://raw.githubusercontent.com/MuhammadBilalFarooq/Assignment1/main/work/notebooks/w03_features.parquet"
features = pd.read_parquet(url)
print(features.shape)
features.head()

(92548, 7)


,client_hash_id,content_hash_id,imp_early,clk_early,pos_early,imp_late,is_declining
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,57.0,0.0,3.659683,20.0,1
1,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,199.0,2.0,4.086084,403.0,0
2,client_62f4a7e64f5e0096,content_275b6f7f733016d4,467.0,1.0,4.449176,343.0,1
3,client_62f4a7e64f5e0096,content_ceaec531566ffcfc,56.0,0.0,6.600595,26.0,1
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,771.0,1.0,1.883472,1087.0,0


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [11]:
import numpy as np

# Early click-through rate (guard against divide-by-zero)
features['ctr_early'] = np.where(
    features['imp_early'] > 0,
    features['clk_early'] / features['imp_early'],
    0
)

min_impressions = 20  # below this, early signal is too thin to trust

# Only consider CTR threshold among pages that got at least SOME clicks
nonzero_ctr = features.loc[features['clk_early'] > 0, 'ctr_early']
ctr_threshold = nonzero_ctr.quantile(0.25)
pos_threshold = features['pos_early'].quantile(0.75)

def assign_reason(row):
    reasons = []
    if row['imp_early'] < min_impressions:
        reasons.append('LOW_VOLUME_UNRELIABLE')
    if row['clk_early'] == 0 and row['imp_early'] >= min_impressions:
        reasons.append('ZERO_CLICKS')
    elif row['clk_early'] > 0 and row['ctr_early'] <= ctr_threshold:
        reasons.append('LOW_CTR')
    if row['pos_early'] >= pos_threshold:
        reasons.append('WEAK_POSITION')
    return reasons if reasons else ['STABLE']

features['reason_codes'] = features.apply(assign_reason, axis=1)
features['baseline_flag'] = features['reason_codes'].apply(lambda r: r != ['STABLE'])

print(f"CTR threshold (bottom 25% of nonzero-click pages): {ctr_threshold:.4f}")
print(f"Position threshold (worst 25%): {pos_threshold:.2f}")
print(f"Flagged: {features['baseline_flag'].sum()} of {len(features)}")


CTR threshold (bottom 25% of nonzero-click pages): 0.0018
Position threshold (worst 25%): 18.04
Flagged: 60563 of 92548


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [12]:
import os

# Simple risk score: count how many reason flags triggered (more flags = more urgent)
features['risk_score'] = features['reason_codes'].apply(
    lambda r: len(r) if r != ['STABLE'] else 0
)

# Action label
def assign_action(row):
    if row['risk_score'] >= 2:
        return 'URGENT_REVIEW'
    elif row['risk_score'] == 1:
        return 'MONITOR'
    else:
        return 'NO_ACTION'

features['action'] = features.apply(assign_action, axis=1)

# Rank: highest risk first, then lowest early position (worse rank) as tiebreaker
ranked = features.sort_values(
    by=['risk_score', 'pos_early'],
    ascending=[False, False]
).reset_index(drop=True)

ranked['rank'] = ranked.index + 1

# Save
os.makedirs('work/outputs', exist_ok=True)
output_cols = ['rank', 'client_hash_id', 'content_hash_id', 'action',
               'risk_score', 'reason_codes', 'imp_early', 'clk_early',
               'pos_early', 'ctr_early']
ranked[output_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)

print(ranked['action'].value_counts())
ranked[output_cols].head(10)


action
MONITOR          42287
NO_ACTION        31985
URGENT_REVIEW    18276
Name: count, dtype: int64


,rank,client_hash_id,content_hash_id,action,risk_score,reason_codes,imp_early,clk_early,pos_early,ctr_early
0,1,client_73cda7b4e4f265ea,content_4a0c4fa4bcc93129,URGENT_REVIEW,2,"[ZERO_CLICKS, WEAK_POSITION]",128.0,0.0,127.620709,0.0
1,2,client_73cda7b4e4f265ea,content_05bb83d0e4179833,URGENT_REVIEW,2,"[ZERO_CLICKS, WEAK_POSITION]",115.0,0.0,102.888475,0.0
2,3,client_73cda7b4e4f265ea,content_767ee3799d993a91,URGENT_REVIEW,2,"[ZERO_CLICKS, WEAK_POSITION]",153.0,0.0,100.098852,0.0
3,4,client_08a6a72ff48e62c0,content_c8d483384985811d,URGENT_REVIEW,2,"[ZERO_CLICKS, WEAK_POSITION]",81.0,0.0,91.842476,0.0
4,5,client_08a6a72ff48e62c0,content_e03809aac3657962,URGENT_REVIEW,2,"[ZERO_CLICKS, WEAK_POSITION]",56.0,0.0,91.305769,0.0
5,6,client_08a6a72ff48e62c0,content_2738cd6cfaeee57f,URGENT_REVIEW,2,"[ZERO_CLICKS, WEAK_POSITION]",205.0,0.0,90.685400,0.0
6,7,client_08a6a72ff48e62c0,content_3e815116fb9fbe6e,URGENT_REVIEW,2,"[ZERO_CLICKS, WEAK_POSITION]",64.0,0.0,90.622540,0.0
7,8,client_62f4a7e64f5e0096,content_108b619046b1a91b,URGENT_REVIEW,2,"[ZERO_CLICKS, WEAK_POSITION]",72.0,0.0,90.611642,0.0
8,9,client_08a6a72ff48e62c0,content_b58d4cf4d1b3c5b6,URGENT_REVIEW,2,"[ZERO_CLICKS, WEAK_POSITION]",118.0,0.0,90.459584,0.0
9,10,client_f623b01661d4bfe4,content_ec882c9a213fcfac,URGENT_REVIEW,2,"[ZERO_CLICKS, WEAK_POSITION]",232.0,0.0,90.302463,0.0


In [13]:
from sklearn.metrics import precision_score, recall_score

flagged = ranked['action'].isin(['URGENT_REVIEW', 'MONITOR'])

precision = precision_score(ranked['is_declining'], flagged)
recall = recall_score(ranked['is_declining'], flagged)

urgent_only = ranked['action'] == 'URGENT_REVIEW'
precision_urgent = precision_score(ranked['is_declining'], urgent_only)

print(f"Baseline (URGENT+MONITOR flagged) — Precision: {precision:.3f}, Recall: {recall:.3f}")
print(f"Baseline (URGENT_REVIEW only)     — Precision: {precision_urgent:.3f}")
print(f"Base rate (share actually declining): {ranked['is_declining'].mean():.3f}")

Baseline (URGENT+MONITOR flagged) — Precision: 0.329, Recall: 0.751
Baseline (URGENT_REVIEW only)     — Precision: 0.314
Base rate (share actually declining): 0.286


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [14]:
top20 = ranked.head(20).copy()
top20['confidence_note'] = top20['reason_codes'].apply(
    lambda r: 'High confidence' if len(r) >= 2 else 'Moderate confidence'
)
top20['what_would_make_it_wrong'] = (
    "If imp_late shows the page recovered impressions despite early weak signals, "
    "or if the page was intentionally deprioritized (e.g. merged/redirected) rather than organically declining."
)

review_cols = ['rank', 'content_hash_id', 'action', 'reason_codes',
               'confidence_note', 'is_declining', 'what_would_make_it_wrong']
top20[review_cols]


,rank,content_hash_id,action,reason_codes,confidence_note,is_declining,what_would_make_it_wrong
0,1,content_4a0c4fa4bcc93129,URGENT_REVIEW,"[ZERO_CLICKS, WEAK_POSITION]",High confidence,1,If imp_late shows the page recovered impressio...
1,2,content_05bb83d0e4179833,URGENT_REVIEW,"[ZERO_CLICKS, WEAK_POSITION]",High confidence,1,If imp_late shows the page recovered impressio...
2,3,content_767ee3799d993a91,URGENT_REVIEW,"[ZERO_CLICKS, WEAK_POSITION]",High confidence,0,If imp_late shows the page recovered impressio...
3,4,content_c8d483384985811d,URGENT_REVIEW,"[ZERO_CLICKS, WEAK_POSITION]",High confidence,0,If imp_late shows the page recovered impressio...
4,5,content_e03809aac3657962,URGENT_REVIEW,"[ZERO_CLICKS, WEAK_POSITION]",High confidence,0,If imp_late shows the page recovered impressio...
5,6,content_2738cd6cfaeee57f,URGENT_REVIEW,"[ZERO_CLICKS, WEAK_POSITION]",High confidence,0,If imp_late shows the page recovered impressio...
6,7,content_3e815116fb9fbe6e,URGENT_REVIEW,"[ZERO_CLICKS, WEAK_POSITION]",High confidence,0,If imp_late shows the page recovered impressio...
7,8,content_108b619046b1a91b,URGENT_REVIEW,"[ZERO_CLICKS, WEAK_POSITION]",High confidence,0,If imp_late shows the page recovered impressio...
8,9,content_b58d4cf4d1b3c5b6,URGENT_REVIEW,"[ZERO_CLICKS, WEAK_POSITION]",High confidence,0,If imp_late shows the page recovered impressio...
9,10,content_ec882c9a213fcfac,URGENT_REVIEW,"[ZERO_CLICKS, WEAK_POSITION]",High confidence,1,If imp_late shows the page recovered impressio...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [15]:
# Weak picks + leakage check

# Leakage check: confirm imp_late and is_declining were never used as inputs to the rule
rule_inputs = ['imp_early', 'clk_early', 'pos_early']
print("Rule inputs used:", rule_inputs)
print("imp_late used in rule?", 'imp_late' in rule_inputs)
print("is_declining used in rule?", 'is_declining' in rule_inputs)

# How many top-20 "high confidence" picks were actually wrong?
wrong_in_top20 = (top20['is_declining'] == 0).sum()
print(f"\nWrong picks in top 20: {wrong_in_top20} of 20 ({wrong_in_top20/20:.0%})")

print(f"""
Weak-pick diagnosis:
- The baseline rule flags pages using ZERO_CLICKS, LOW_CTR, and WEAK_POSITION,
  computed only from early-March data (imp_early, clk_early, pos_early).
- Overall precision (0.329) is barely above the base rate (0.286), meaning the
  rule performs only marginally better than random flagging.
- The single largest reason-code group, ZERO_CLICKS, catches pages that already
  had almost no visibility in early March. These pages read as "at risk" by the
  rule, but many of them were already flat rather than actively declining --
  there is little room left to decline further, so the rule cannot reliably
  tell "already low" apart from "getting worse."
- WEAK_POSITION alone shows a similar pattern: a poor early rank correlates
  weakly with later decline, but many poorly-ranked pages simply stay poorly
  ranked rather than declining further.
- Conclusion: early click and position signals, on their own, are weak
  predictors of decline in this dataset. This sets an honest, low baseline
  (~0.33 precision) for the model in the next section to try to beat.
""")


Rule inputs used: ['imp_early', 'clk_early', 'pos_early']
imp_late used in rule? False
is_declining used in rule? False

Wrong picks in top 20: 13 of 20 (65%)

Weak-pick diagnosis:
- The baseline rule flags pages using ZERO_CLICKS, LOW_CTR, and WEAK_POSITION,
  computed only from early-March data (imp_early, clk_early, pos_early).
- Overall precision (0.329) is barely above the base rate (0.286), meaning the
  rule performs only marginally better than random flagging.
- The single largest reason-code group, ZERO_CLICKS, catches pages that already
  had almost no visibility in early March. These pages read as "at risk" by the
  rule, but many of them were already flat rather than actively declining --
  there is little room left to decline further, so the rule cannot reliably
  tell "already low" apart from "getting worse."
- WEAK_POSITION alone shows a similar pattern: a poor early rank correlates
  weakly with later decline, but many poorly-ranked pages simply stay poorly
  ranked rat

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

Every section above is filled ✅
Notebook runs top to bottom with no errors ✅
No client names, URLs, or private queries anywhere ✅ (we only used hashed IDs)
Claims use careful words: observed, measured, directional, decision-support ✅ (your diagnosis text already does this)